In [2]:
%pip install pandas xlrd openpyxl

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 96 kB 9.0 MB/s  eta 0:00:01
     |████████████████████████████████| 250 kB 4.5 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd

GPR_FILE = "data_gpr_daily_recent.xls"   # change to your exact file name
START = "2015-01-01"
OUT_FILE = "gpr_daily_2015_2026.csv"

In [5]:
import pandas as pd
from pathlib import Path

# --- Folders (the notebook runs from war-puzzle-revisited/notebooks) ---
PROJECT = Path.cwd().parent
RAW = PROJECT / "data" / "raw"
PROCESSED = PROJECT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

# --- Settings ---
GPR_FILE = RAW / "data_gpr_daily_recent.xls"
START = "2015-01-01"
OUT_FILE = PROCESSED / "gpr_daily_2015_2026.csv"

print("Reading from:", GPR_FILE, "| exists:", GPR_FILE.exists())

Reading from: /Users/abrar/war-puzzle-revisited/data/raw/data_gpr_daily_recent.xls | exists: True


In [8]:
gpr = gpr_raw.copy()
gpr["date"] = pd.to_datetime(gpr["date"])

gpr = gpr[["date", "GPRD", "GPRD_ACT", "GPRD_THREAT", "N10D", "event"]]
gpr = gpr[gpr["date"] >= START].sort_values("date").reset_index(drop=True)
gpr.head()

,date,GPRD,GPRD_ACT,GPRD_THREAT,N10D,event
0,2015-01-01,74.579384,49.566547,82.686790,533,NaN
1,2015-01-02,106.857018,117.180916,97.740456,620,NaN
2,2015-01-03,53.428509,60.873203,44.427479,868,NaN
3,2015-01-04,16.480436,16.429708,13.703999,402,NaN
4,2015-01-05,100.719963,111.566589,74.446053,592,NaN


In [9]:
print("Rows:", len(gpr), "| From", gpr["date"].min().date(), "to", gpr["date"].max().date())
print("\nMissing values:\n", gpr.isna().sum())
gpr.describe()

Rows: 4310 | From 2015-01-01 to 2026-10-19

Missing values:
 date              0
GPRD              0
GPRD_ACT          0
GPRD_THREAT       0
N10D              0
event          4307
dtype: int64


,date,GPRD,GPRD_ACT,GPRD_THREAT,N10D
count,4310,4310.000000,4310.000000,4310.000000,4310.000000
mean,2020-11-24 12:00:00,114.503460,103.133847,129.210307,633.934571
min,2015-01-01 00:00:00,0.000000,0.000000,0.000000,151.000000
25%,2017-12-13 06:00:00,74.161964,50.297899,77.778383,522.000000
50%,2020-11-24 12:00:00,103.780991,88.063240,114.489410,624.000000
75%,2023-11-06 18:00:00,142.738449,136.802097,162.459301,737.000000
max,2026-10-19 00:00:00,540.827393,627.332214,809.486816,1169.000000
std,NaN,60.450040,75.907357,76.972757,162.901531


In [10]:
gpr.to_csv(OUT_FILE, index=False)
print("Saved", OUT_FILE)

Saved /Users/abrar/war-puzzle-revisited/data/processed/gpr_daily_2015_2026.csv


In [11]:
print(gpr.nlargest(10, "GPRD")[["date", "GPRD", "GPRD_THREAT", "GPRD_ACT"]])
print("\nLabelled events:")
print(gpr.dropna(subset=["event"])[["date", "event"]])

           date        GPRD  GPRD_THREAT    GPRD_ACT
2616 2022-03-01  540.827393   809.486816  287.553436
3826 2025-06-23  540.163330   606.099060  583.916321
2617 2022-03-02  523.524658   778.195496  203.222855
2612 2022-02-25  515.891663   797.752441  162.411713
4078 2026-03-02  500.807159   602.077332  553.402832
2631 2022-03-16  473.223969   632.542908  343.905151
4080 2026-03-04  465.918640   548.065918  611.760071
4086 2026-03-10  457.110596   434.402039  627.332214
2615 2022-02-28  452.618073   678.286255  218.174667
4107 2026-03-31  447.822266   492.853546  590.881531

Labelled events:
           date                           event
320  2015-11-17         Paris Terrorist Attacks
1832 2020-01-07   U.S. / Iran Tensions Escalate
2612 2022-02-25       Russia / Ukraine Tensions


In [13]:
%pip install yfinance

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [16]:
import yfinance as yf
import time
from datetime import date
print("yfinance version:", yf.__version__)
TODAY = date.today().isoformat()
END = (pd.Timestamp.today() + pd.Timedelta(days=1)).strftime("%Y-%m-%d")   # +1 day so today is included

TICKERS = [
    "^GSPC", "SPY", "ACWX", "^VIX",
    "XLE", "XLK", "XLF", "XLV", "XLY", "XLP", "XLI", "XLB", "XLU", "XLRE", "XLC",
    "ITA", "JETS",
]

series = {}
failed = []
for t in TICKERS:
    try:
        df = yf.download(t, start=START, end=END, auto_adjust=True, progress=False)
        series[t] = df["Close"].squeeze()
        print(f"{t:6s} OK   from {series[t].first_valid_index().date()}  ({len(series[t])} rows)")
    except Exception as e:
        failed.append(t)
        print(f"{t:6s} FAILED: {e}")
    time.sleep(1)   # small pause so Yahoo doesn't block us

prices = pd.DataFrame(series)
prices.index.name = "date"
print("\nFailed:", failed)
print(prices.shape)

prices.to_csv(RAW / f"prices_yahoo_{TODAY}.csv")

yfinance version: 0.2.64
^GSPC  OK   from 2015-01-02  (2967 rows)
SPY    OK   from 2015-01-02  (2967 rows)
ACWX   OK   from 2015-01-02  (2967 rows)
^VIX   OK   from 2015-01-02  (2969 rows)
XLE    OK   from 2015-01-02  (2967 rows)
XLK    OK   from 2015-01-02  (2967 rows)
XLF    OK   from 2015-01-02  (2967 rows)
XLV    OK   from 2015-01-02  (2967 rows)
XLY    OK   from 2015-01-02  (2967 rows)
XLP    OK   from 2015-01-02  (2967 rows)
XLI    OK   from 2015-01-02  (2967 rows)
XLB    OK   from 2015-01-02  (2967 rows)
XLU    OK   from 2015-01-02  (2967 rows)
XLRE   OK   from 2015-10-08  (2774 rows)
XLC    OK   from 2018-06-19  (2096 rows)
ITA    OK   from 2015-01-02  (2967 rows)
JETS   OK   from 2015-04-30  (2886 rows)

Failed: []
(2969, 17)


In [17]:
import numpy as np

px = prices.dropna(subset=["SPY"]).copy()        # SPY trading days = our calendar
px.index = pd.to_datetime(px.index)

ret = 100 * np.log(px.drop(columns=["^VIX"])).diff()
ret.columns = ["RET_" + c.replace("^", "") for c in ret.columns]   # e.g. RET_GSPC, RET_XLE

df = ret.copy()
df["VIX"] = px["^VIX"]
df["VIX_CHG"] = px["^VIX"].diff()
df.index.name = "date"
df.head()

,RET_GSPC,RET_SPY,RET_ACWX,RET_XLE,RET_XLK,RET_XLF,RET_XLV,RET_XLY,RET_XLP,RET_XLI,RET_XLB,RET_XLU,RET_XLRE,RET_XLC,RET_ITA,RET_JETS,VIX,VIX_CHG
date,,,,,,,,,,,,,,,,,,
2015-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17.790001,NaN
2015-01-05,-1.844721,-1.822464,-2.198921,-4.224792,-1.538280,-2.125130,-0.511318,-1.931133,-0.705689,-2.363591,-2.581827,-1.230109,NaN,NaN,-1.496091,NaN,19.920000,2.129999
2015-01-06,-0.893325,-0.946354,-1.033294,-1.479914,-1.213053,-1.540104,-0.337408,-1.001321,-0.125061,-1.239752,-0.889866,0.063978,NaN,NaN,-0.640457,NaN,21.120001,1.200001
2015-01-07,1.156274,1.238414,1.128881,0.212731,0.843263,1.043190,2.323986,1.568977,1.695628,0.749333,1.121571,0.976232,NaN,NaN,0.879537,NaN,19.309999,-1.810001
2015-01-08,1.773017,1.758915,1.446347,2.220131,2.174264,1.483335,1.694092,1.489042,1.505936,1.983114,2.350122,0.694506,NaN,NaN,1.944709,NaN,17.010000,-2.299999


In [19]:
oil = pd.read_csv(
    "https://fred.stlouisfed.org/graph/fredgraph.csv?id=DCOILWTICO",
    na_values=".", parse_dates=["observation_date"]
).rename(columns={"observation_date": "date", "DCOILWTICO": "WTI"})

oil = oil[oil["date"] >= START]
print(oil.tail())
oil.to_csv(RAW / f"oil_wti_fred_{TODAY}.csv", index=False)

            date     WTI
10635 2026-10-08   98.26
10636 2026-10-09   98.73
10637 2026-10-12   98.73
10638 2026-10-13  100.86
10639 2026-10-14  101.91


In [20]:
FF_URL = ("https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"
          "F-F_Research_Data_5_Factors_2x3_daily_CSV.zip")

ff = pd.read_csv(FF_URL, skiprows=3, index_col=0)
ff.index = pd.to_datetime(ff.index.astype(str).str.strip(), format="%Y%m%d", errors="coerce")
ff = ff[ff.index.notna()].apply(pd.to_numeric, errors="coerce")
ff.index.name = "date"
ff = ff[ff.index >= START]

print(ff.tail())   # values are in percent
ff.to_csv(RAW / f"ff5_daily_{TODAY}.csv")

            Mkt-RF   SMB   HML   RMW   CMA    RF
date                                            
2026-08-25    0.31  0.00 -0.33 -0.88 -0.30  0.01
2026-08-26   -0.06 -0.13  0.20  0.26  0.46  0.01
2026-08-27    0.72 -0.17 -1.14 -0.78 -0.89  0.01
2026-08-28   -0.34 -0.37  0.28  1.66  0.49  0.01
2026-08-31   -0.33 -0.29 -0.39 -1.09  0.03  0.01


In [21]:
wti = oil.set_index("date")["WTI"].reindex(df.index)
df["WTI"] = wti
df["RET_WTI"] = 100 * np.log(wti.where(wti > 0)).diff()   # WTI went negative on 20 Apr 2020 -> left missing

In [22]:
ff_clean = ff.rename(columns={"Mkt-RF": "MKT_RF"})
df = df.join(ff_clean, how="left")

In [23]:
g = gpr[["date", "GPRD", "GPRD_ACT", "GPRD_THREAT"]].sort_values("date")
cal = pd.DataFrame({"trade_date": df.index})

# each calendar day -> the next trading day on or after it
g = pd.merge_asof(g, cal, left_on="date", right_on="trade_date", direction="forward")
g_td = g.groupby("trade_date")[["GPRD", "GPRD_ACT", "GPRD_THREAT"]].mean()

# log(1 + GPR) levels and daily changes
for col, short in [("GPRD", "GPR"), ("GPRD_THREAT", "GPRT"), ("GPRD_ACT", "GPRA")]:
    g_td[f"L{short}"] = np.log1p(g_td[col])
    g_td[f"D{short}"] = g_td[f"L{short}"].diff()

df = df.join(g_td)

In [24]:
print("Shape:", df.shape)
print("From", df.index.min().date(), "to", df.index.max().date())
print("Last Fama-French date:", df["MKT_RF"].last_valid_index().date())
print("Last oil date:", df["WTI"].last_valid_index().date())
print("\nMissing values per column:\n", df.isna().sum())

# check the weekend handling around 7 Oct 2023 (a Saturday)
print(df.loc["2023-10-05":"2023-10-11", ["RET_GSPC", "GPRD", "GPRD_THREAT", "GPRD_ACT", "DGPRA"]])

Shape: (2967, 35)
From 2015-01-02 to 2026-10-20
Last Fama-French date: 2026-08-31
Last oil date: 2026-10-14

Missing values per column:
 RET_GSPC         1
RET_SPY          1
RET_ACWX         1
RET_XLE          1
RET_XLK          1
RET_XLF          1
RET_XLV          1
RET_XLY          1
RET_XLP          1
RET_XLI          1
RET_XLB          1
RET_XLU          1
RET_XLRE       194
RET_XLC        872
RET_ITA          1
RET_JETS        82
VIX              0
VIX_CHG          1
WTI             26
RET_WTI         48
MKT_RF          35
SMB             35
HML             35
RMW             35
CMA             35
RF              35
GPRD             1
GPRD_ACT         1
GPRD_THREAT      1
LGPR             1
DGPR             2
LGPRT            1
DGPRT            2
LGPRA            1
DGPRA            2
dtype: int64
            RET_GSPC        GPRD  GPRD_THREAT    GPRD_ACT     DGPRA
date                                                               
2023-10-05 -0.130488   54.304386    70.242355   3

In [25]:
df.to_csv(PROCESSED / "daily_dataset.csv")
print("Saved", PROCESSED / "daily_dataset.csv")

Saved /Users/abrar/war-puzzle-revisited/data/processed/daily_dataset.csv
